In [3]:
# Indian Skincare Demand Intelligence — Data Collection & Normalisation

"""
This notebook collects Google Trends search data for 29 terms across 4 buckets 
(ingredients, product types, skin concerns, brand searches) and normalises them 
into a single comparable dataset using an anchor term methodology.

Data: Google Trends, geo-filtered to India, July 2021 – June 2026  
Output: skincare_trends_master.xlsx (61 months × 31 terms)
"""

'\nThis notebook collects Google Trends search data for 29 terms across 4 buckets \n(ingredients, product types, skin concerns, brand searches) and normalises them \ninto a single comparable dataset using an anchor term methodology.\n\nData: Google Trends, geo-filtered to India, July 2021 – June 2026  \nOutput: skincare_trends_master.xlsx (61 months × 31 terms)\n'

In [3]:
!pip install pytrends
!pip install pandas
!pip install openpyxl

In [7]:
!pip show pytrends

Name: pytrends
Version: 4.9.2
Summary: Pseudo API for Google Trends
Home-page: 
Author: John Hogue, Burton DeWilde
Author-email: dreyco676@gmail.com
License: Apache 2.0
Location: C:\Users\Saanvi\anaconda3\Lib\site-packages
Requires: lxml, pandas, requests
Required-by: 


In [5]:
## Step 1 : Anchor Term Selection

'''
Google Trends returns relative scores (0–100) within each pull of 5 terms. 
To compare terms across batches, we need one common anchor term in every batch.

We select the anchor by testing candidate terms on two criteria:
- **Coefficient of Variation (CV)** — lower = more stable over time = better anchor
- **Mean value** — should be in 30–60 range so it doesn't compress other terms

The term with lowest CV among neutral candidates becomes the anchor.
'''

"\nGoogle Trends returns relative scores (0–100) within each pull of 5 terms. \nTo compare terms across batches, we need one common anchor term in every batch.\n\nWe select the anchor by testing candidate terms on two criteria:\n- **Coefficient of Variation (CV)** — lower = more stable over time = better anchor\n- **Mean value** — should be in 30–60 range so it doesn't compress other terms\n\nThe term with lowest CV among neutral candidates becomes the anchor.\n"

In [11]:
import pandas as pd
import numpy as np

# Load batch 2 as test
df = pd.read_csv('batch_2.csv')

# Calculate mean and std for each term
stats = pd.DataFrame({
    'mean': df.iloc[:, 1:].mean(),
    'std': df.iloc[:, 1:].std(),
    'cv': df.iloc[:, 1:].std() / df.iloc[:, 1:].mean()  # coefficient of variation
})

print(stats.sort_values('cv'))  # lowest cv = most stable = best anchor candidate

                  mean        std        cv
Vitamin C    36.213115   5.600937  0.154666
face wash    67.409836  11.499822  0.170596
moisturiser   2.196721   0.476783  0.217043
face serum   12.704918   2.979621  0.234525
Niacinamide  10.508197   4.577565  0.435618


In [7]:
## Step 2 : Load All Batches

'''
Data was manually exported from Google Trends in 8 batches of 5 terms each.
Vitamin C is included in every batch as the anchor term.
'''

'\nData was manually exported from Google Trends in 8 batches of 5 terms each.\nVitamin C is included in every batch as the anchor term.\n'

In [27]:
import pandas as pd
import numpy as np

ANCHOR = 'Vitamin C'

batch_files = {
    1: 'batch_1.csv',
    2: 'batch_2.csv',
    3: 'batch_3.csv',
    4: 'batch_4.csv',
    5: 'batch_5.csv',
    6: 'batch_6.csv',
    7: 'batch_7.csv',
    8: 'batch_8.csv',
}

raw_batches = {}
for i, file in batch_files.items():
    df = pd.read_csv(file)
    df.columns = df.columns.str.strip()
    df['Time'] = pd.to_datetime(df['Time'])
    df = df.set_index('Time')
    df = df.apply(pd.to_numeric, errors='coerce')
    raw_batches[i] = df
    print(f"Batch {i}: {list(df.columns)}")

print("\nAll batches loaded.")

Batch 1: ['Vitamin C', 'niacinamide', 'retinol', 'hyaluronic acid', 'face wash']
Batch 2: ['Vitamin C', 'kojic acid', 'salicylic acid', 'tranexamic acid', 'ceramides']
Batch 3: ['Vitamin C', 'rice water', 'face serum', 'moisturiser', 'toner']
Batch 4: ['Vitamin C', 'sunscreen', 'exfoliator', 'lip balm', 'face mask']
Batch 5: ['Vitamin C', 'Acne', 'Dark Circles', 'Dark Spots', 'Skin Brightening']
Batch 6: ['Vitamin C', 'Oily Skin', 'Tan removal', 'Minimalist', 'Foxtale']
Batch 7: ['Vitamin C', 'Dot and Key', 'Plum', 'Pilgrim', 'Lakme']
Batch 8: ['Vitamin C', 'Himalaya Wellness Company', 'Cetaphil']

All batches loaded.


In [9]:
## Step 3 : Normalise Against Anchor

'''
Each term's value is divided by the anchor's mean in that batch and multiplied 
by 100. This puts all 29 terms on the same scale so cross-batch comparison is valid.
'''

"\nEach term's value is divided by the anchor's mean in that batch and multiplied \nby 100. This puts all 29 terms on the same scale so cross-batch comparison is valid.\n"

In [29]:
# Normalise all batches against Vitamin C anchor
def normalise_batch(df, anchor=ANCHOR):
    anchor_mean = df[anchor].mean()
    normalised = pd.DataFrame(index=df.index)
    for col in df.columns:
        if col != anchor:
            normalised[col] = (df[col] / anchor_mean) * 100
    return normalised

# Apply to all batches and combine
normalised_dfs = []
for i, df in raw_batches.items():
    norm_df = normalise_batch(df)
    normalised_dfs.append(norm_df)

# Add Vitamin C itself from batch 1 (anchor, score = 100 baseline)
anchor_df = pd.DataFrame(
    raw_batches[1][ANCHOR] / raw_batches[1][ANCHOR].mean() * 100,
    columns=[ANCHOR]
)
normalised_dfs.append(anchor_df)

# Merge everything
master_df = pd.concat(normalised_dfs, axis=1)
master_df = master_df.sort_index()

print(master_df.shape)
print(master_df.head())

(61, 31)
            niacinamide    retinol  hyaluronic acid   face wash  kojic acid  \
Time                                                                          
2021-06-01    10.966292  13.707865         8.224719  134.337079    4.021978   
2021-07-01     8.224719  10.966292         8.224719  126.112360    2.681319   
2021-08-01    10.966292  10.966292         8.224719  128.853933    2.681319   
2021-09-01    10.966292  13.707865         5.483146  128.853933    2.681319   
2021-10-01     8.224719  10.966292         5.483146  123.370787    2.681319   

            salicylic acid  tranexamic acid  ceramides  rice water  \
Time                                                                 
2021-06-01       26.813187         5.362637        0.0   18.769231   
2021-07-01       22.791209         5.362637        0.0   17.428571   
2021-08-01       21.450549         5.362637        0.0   13.406593   
2021-09-01       18.769231         5.362637        0.0   13.406593   
2021-10-01       

In [31]:
# Define buckets for separate sheets
ingredients = ['niacinamide', 'retinol', 'hyaluronic acid', 'Vitamin C',
               'kojic acid', 'salicylic acid', 'tranexamic acid', 'ceramides', 'rice water']

products = ['face wash', 'face serum', 'moisturiser', 'toner', 'sunscreen',
            'exfoliator', 'lip balm', 'face mask', 'under-eye cream']

concerns = ['dark spots', 'acne treatment', 'skin brightening',
            'Oily Skin', 'Tan removal']

brands = ['Minimalist', 'Foxtale', 'Dot and Key', 'Plum',
          'Pilgrim', 'Lakme', 'Glow and Lovely', 'Cetaphil']

# Export
with pd.ExcelWriter('skincare_trends_master.xlsx', engine='openpyxl') as writer:
    master_df.to_excel(writer, sheet_name='All Terms')
    master_df[[c for c in ingredients if c in master_df.columns]].to_excel(writer, sheet_name='Ingredients')
    master_df[[c for c in products if c in master_df.columns]].to_excel(writer, sheet_name='Products')
    master_df[[c for c in concerns if c in master_df.columns]].to_excel(writer, sheet_name='Concerns')
    master_df[[c for c in brands if c in master_df.columns]].to_excel(writer, sheet_name='Brands')

print("Exported to skincare_trends_final.xlsx")

Exported to skincare_trends_final.xlsx


In [33]:
# Normalise all batches against Vitamin C anchor
def normalise_batch(df, anchor=ANCHOR):
    anchor_mean = df[anchor].mean()
    normalised = pd.DataFrame(index=df.index)
    for col in df.columns:
        if col != anchor:
            normalised[col] = (df[col] / anchor_mean) * 100
    return normalised

# Apply to all batches and combine
normalised_dfs = []
for i, df in raw_batches.items():
    norm_df = normalise_batch(df)
    normalised_dfs.append(norm_df)

# Add Vitamin C itself from batch 1 (anchor, score = 100 baseline)
anchor_df = pd.DataFrame(
    raw_batches[1][ANCHOR] / raw_batches[1][ANCHOR].mean() * 100,
    columns=[ANCHOR]
)
normalised_dfs.append(anchor_df)

# Merge everything
master_df = pd.concat(normalised_dfs, axis=1)
master_df = master_df.sort_index()

print(master_df.shape)
print(master_df.head())

(61, 31)
            niacinamide    retinol  hyaluronic acid   face wash  kojic acid  \
Time                                                                          
2021-06-01    10.966292  13.707865         8.224719  134.337079    4.021978   
2021-07-01     8.224719  10.966292         8.224719  126.112360    2.681319   
2021-08-01    10.966292  10.966292         8.224719  128.853933    2.681319   
2021-09-01    10.966292  13.707865         5.483146  128.853933    2.681319   
2021-10-01     8.224719  10.966292         5.483146  123.370787    2.681319   

            salicylic acid  tranexamic acid  ceramides  rice water  \
Time                                                                 
2021-06-01       26.813187         5.362637        0.0   18.769231   
2021-07-01       22.791209         5.362637        0.0   17.428571   
2021-08-01       21.450549         5.362637        0.0   13.406593   
2021-09-01       18.769231         5.362637        0.0   13.406593   
2021-10-01       

In [29]:
# Check which product terms have highest normalised search volume
products = ['face wash', 'face serum', 'moisturiser', 'toner', 'sunscreen',
            'exfoliator', 'lip balm', 'face mask', 'under-eye cream']

product_means = master_df[[c for c in products if c in master_df.columns]].mean().sort_values(ascending=False)
print(product_means.round(2))

face wash          185.53
sunscreen          151.76
toner               42.64
face mask           41.76
lip balm            38.99
face serum          34.90
moisturiser          6.07
exfoliator           0.08
under-eye cream      0.00
dtype: float64


In [35]:
import pandas as pd
import numpy as np

nykaa_df = pd.read_excel('nykaa_data.xlsx')
print(nykaa_df.shape)
print(nykaa_df.head())

(32, 8)
        Brand    Category  Num_SKUs  Price_Min  Price_Max  BestSeller_Price  \
0  Minimalist   Face Wash         6        284        569            284.00   
1  Minimalist   Sunscreen         6        379        569            426.50   
2  Minimalist       Toner         4        379        474            474.00   
3  Minimalist  Face Serum        23        237        664            248.75   
4     Foxtale   Face Wash         6        212        324            290.00   

   Avg_Rating  Num_Reviews (Avg)  
0         4.0            2634.30  
1         4.0            4927.00  
2         4.0            1289.00  
3         4.0            4854.75  
4         4.0            1167.67  


In [37]:
# Load Google Trends master data
trends_df = pd.read_excel('skincare_trends_master.xlsx', sheet_name='All Terms')
trends_df = trends_df.set_index('Time')

print(trends_df.shape)
print(trends_df.head())

(61, 31)
            niacinamide    retinol  hyaluronic acid   face wash  kojic acid  \
Time                                                                          
2021-06-01    10.966292  13.707865         8.224719  134.337079    4.021978   
2021-07-01     8.224719  10.966292         8.224719  126.112360    2.681319   
2021-08-01    10.966292  10.966292         8.224719  128.853933    2.681319   
2021-09-01    10.966292  13.707865         5.483146  128.853933    2.681319   
2021-10-01     8.224719  10.966292         5.483146  123.370787    2.681319   

            salicylic acid  tranexamic acid  ceramides  rice water  \
Time                                                                 
2021-06-01       26.813187         5.362637        0.0   18.769231   
2021-07-01       22.791209         5.362637        0.0   17.428571   
2021-08-01       21.450549         5.362637        0.0   13.406593   
2021-09-01       18.769231         5.362637        0.0   13.406593   
2021-10-01       

In [39]:
# Brand search volume trends
brands = ['Minimalist', 'Foxtale', 'Dot and Key', 'Plum', 
          'Pilgrim', 'Lakme', 'Himalaya Wellness Company', 'Cetaphil']

# Filter trends_df for brand columns only
brand_trends = trends_df[[c for c in brands if c in trends_df.columns]]

# Check which brands were found
print("Brands found in trends data:", list(brand_trends.columns))
print("\nBrand mean search volumes (normalised):")
print(brand_trends.mean().sort_values(ascending=False).round(2))

# Growth rate — compare first year avg vs last year avg
first_year = brand_trends.iloc[:12].mean()
last_year = brand_trends.iloc[-12:].mean()
growth = ((last_year - first_year) / first_year * 100).round(1)

print("\nBrand growth (first year vs last year %):")
print(growth.sort_values(ascending=False))

Brands found in trends data: ['Minimalist', 'Foxtale', 'Dot and Key', 'Plum', 'Pilgrim', 'Lakme', 'Himalaya Wellness Company', 'Cetaphil']

Brand mean search volumes (normalised):
Lakme                        61.54
Plum                         53.19
Cetaphil                     43.57
Minimalist                   34.95
Pilgrim                      25.58
Dot and Key                  20.09
Himalaya Wellness Company    19.76
Foxtale                      10.07
dtype: float64

Brand growth (first year vs last year %):
Foxtale                         inf
Dot and Key                  1273.1
Pilgrim                       670.0
Minimalist                    275.4
Cetaphil                      200.0
Plum                           55.0
Lakme                         -34.2
Himalaya Wellness Company     -85.4
dtype: float64


In [41]:
# Ingredient search volume trends
ingredients = ['niacinamide', 'retinol', 'hyaluronic acid', 'Vitamin C',
               'kojic acid', 'salicylic acid', 'tranexamic acid', 'ceramides', 'rice water']

ingredient_trends = trends_df[[c for c in ingredients if c in trends_df.columns]]

print("Ingredient mean search volumes:")
print(ingredient_trends.mean().sort_values(ascending=False).round(2))

first_year = ingredient_trends.iloc[:12].mean()
last_year = ingredient_trends.iloc[-12:].mean()
growth = ((last_year - first_year) / first_year * 100).round(1)

print("\nIngredient growth (first year vs last year %):")
print(growth.sort_values(ascending=False))

Ingredient mean search volumes:
Vitamin C          100.00
salicylic acid      34.88
niacinamide         29.08
retinol             21.66
rice water          18.11
kojic acid          12.42
hyaluronic acid     12.00
tranexamic acid      7.27
ceramides            1.01
dtype: float64

Ingredient growth (first year vs last year %):
ceramides            inf
kojic acid         390.9
niacinamide        308.3
retinol            112.7
salicylic acid      97.9
rice water          87.9
hyaluronic acid     60.0
Vitamin C           49.0
tranexamic acid     35.8
dtype: float64


In [43]:
# Skin concerns trends
concerns = ['Dark Spots', 'Acne', 'Skin Brightening', 'Oily Skin', 'Tan removal']

concern_trends = trends_df[[c for c in concerns if c in trends_df.columns]]

print("Concern mean search volumes:")
print(concern_trends.mean().sort_values(ascending=False).round(2))

first_year = concern_trends.iloc[:12].mean()
last_year = concern_trends.iloc[-12:].mean()
growth = ((last_year - first_year) / first_year * 100).round(1)

print("\nConcern growth (first year vs last year %):")
print(growth.sort_values(ascending=False))

Concern mean search volumes:
Acne                105.18
Oily Skin            51.98
Dark Spots           15.50
Tan removal          11.85
Skin Brightening      6.16
dtype: float64

Concern growth (first year vs last year %):
Skin Brightening    151.4
Tan removal         137.9
Oily Skin            52.8
Acne                 34.0
Dark Spots           12.6
dtype: float64


In [45]:
# Supply vs Demand gap analysis
sku_by_brand = nykaa_df.groupby('Brand')['Num_SKUs'].sum().sort_values(ascending=False)
print("Total SKUs by brand:")
print(sku_by_brand)

price_by_brand = nykaa_df.groupby('Brand')['BestSeller_Price'].mean().round(0).sort_values(ascending=False)
print("\nAverage bestseller price by brand (₹):")
print(price_by_brand)

reviews_by_brand = nykaa_df.groupby('Brand')['Num_Reviews (Avg)'].mean().round(0).sort_values(ascending=False)
print("\nAverage reviews by brand:")
print(reviews_by_brand)

Total SKUs by brand:
Brand
Lakme         40
Plum          40
Minimalist    39
Pilgrim       39
Dot & Key     34
Himalya       31
Foxtale       23
Cetaphil      19
Name: Num_SKUs, dtype: int64

Average bestseller price by brand (₹):
Brand
Cetaphil      923.0
Plum          444.0
Foxtale       407.0
Minimalist    358.0
Dot & Key     342.0
Pilgrim       339.0
Lakme         325.0
Himalya       310.0
Name: BestSeller_Price, dtype: float64

Average reviews by brand:
Brand
Minimalist    3426.0
Plum          2932.0
Dot & Key     2696.0
Cetaphil      1844.0
Foxtale        990.0
Lakme          942.0
Himalya        787.0
Pilgrim        656.0
Name: Num_Reviews (Avg), dtype: float64
